In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Load latest session
files = sorted(Path('../data/raw').glob('session_*.npz'))
latest = files[-1]
d = np.load(latest)
eeg = d['eeg']      # (40, 8, 1000)
labels = d['labels']

print(f"Loaded: {latest.name}")
print(f"Shape: {eeg.shape}  |  Left: {(labels==0).sum()}  Right: {(labels==1).sum()}")

In [ ]:
# Channels 3 and 4 are 0-indexed as 2 and 3
CH3_IDX = 2
CH4_IDX = 3
SRATE = 250
time_axis = np.linspace(0.5, 4.5, 1000)  # seconds post-cue (epoch window)

left_trials  = eeg[labels == 0]   # (20, 8, 1000)
right_trials = eeg[labels == 1]   # (20, 8, 1000)

def plot_trial(arm, trial_num):
    trials = left_trials if arm == 'Left (Ch4 = motor)' else right_trials
    label  = 'Left arm' if arm == 'Left (Ch4 = motor)' else 'Right arm'
    idx = trial_num - 1

    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
    fig.suptitle(f"{label}  —  Trial {trial_num}/20", fontsize=14, fontweight='bold')

    for ax, ch_idx, ch_name, color in zip(
        axes,
        [CH3_IDX, CH4_IDX],
        ['Ch 3 (C4 — right motor cortex)', 'Ch 4 (C3 — left motor cortex)'],
        ['#E05A44', '#4682DC']
    ):
        signal = trials[idx, ch_idx, :]
        ax.plot(time_axis, signal, color=color, linewidth=0.9)
        ax.set_ylabel('Amplitude (µV)', fontsize=10)
        ax.set_title(ch_name, fontsize=11)
        ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, label='epoch start')
        ax.grid(True, alpha=0.3)
        peak = np.max(np.abs(signal))
        ax.set_ylim(-peak*1.3, peak*1.3)

    axes[-1].set_xlabel('Time post-cue (s)', fontsize=10)
    plt.tight_layout()
    plt.show()

arm_widget   = widgets.ToggleButtons(
    options=['Left (Ch4 = motor)', 'Right (Ch3 = motor)'],
    description='Arm:',
    button_style='info'
)
trial_slider = widgets.IntSlider(
    value=1, min=1, max=20, step=1,
    description='Trial:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

out = widgets.interactive_output(plot_trial, {'arm': arm_widget, 'trial_num': trial_slider})
display(widgets.VBox([arm_widget, trial_slider]), out)

In [ ]:
# Overview: all 20 trials for each arm on Ch3 and Ch4
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
fig.suptitle('All 20 trials — Ch3 & Ch4  (faint = individual, bold = mean)', fontsize=13)

combos = [
    (left_trials,  CH3_IDX, 'Left arm  —  Ch3', '#E05A44', axes[0,0]),
    (left_trials,  CH4_IDX, 'Left arm  —  Ch4', '#4682DC', axes[1,0]),
    (right_trials, CH3_IDX, 'Right arm  —  Ch3', '#E05A44', axes[0,1]),
    (right_trials, CH4_IDX, 'Right arm  —  Ch4', '#4682DC', axes[1,1]),
]

for trials, ch_idx, title, color, ax in combos:
    for t in range(20):
        ax.plot(time_axis, trials[t, ch_idx, :], color=color, alpha=0.2, linewidth=0.7)
    ax.plot(time_axis, trials[:, ch_idx, :].mean(axis=0), color=color, linewidth=2.0, label='mean')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('µV')
    ax.grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel('Time post-cue (s)')

plt.tight_layout()
plt.show()